# 第16章 信用风险与违约预测 —— 配套代码

对应正文 `book/part4/16-credit-risk.md`。

> **运行前准备**：请先在终端执行 `uv run python scripts/make_sample_data.py` 生成内置示例数据。

## 演示内容

1. 环境初始化与数据加载
2. 探索性分析（EDA）：违约率与特征分布
3. WOE/IV 计算与变量筛选
4. WOE 分箱可视化
5. 逻辑回归评分卡训练
6. ROC/AUC + KS 曲线评估
7. 混淆矩阵与分类报告
8. 类别不平衡处理：class_weight 对比
9. 评分卡刻度：概率转信用分
10. XGBoost 对比（可选）
11. 模型汇总
12. 习题参考解答

In [ ]:
# === 在线运行引导（Google Colab）。本地 / Binder 会自动跳过 ===
import sys

if "google.colab" in sys.modules:
    import os
    import subprocess

    if not os.path.isdir("src/fds"):  # 尚未进入仓库
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/albertandking/financial-data-science", "fds_book"],
                       check=True)
        os.chdir("fds_book")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
        print("已就绪：仓库已克隆、fds 已安装、内置数据随仓库一并克隆。")

In [ ]:
# Cell 1：环境初始化与数据加载
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, roc_curve, classification_report,
    ConfusionMatrixDisplay, confusion_matrix
)
from sklearn.preprocessing import StandardScaler

from fds import load_credit, set_chinese_font

set_chinese_font()
SEED = 42
np.random.seed(SEED)

df = load_credit()

FEATURE_COLS = ['age', 'income', 'debt_to_income',
                'credit_history_months', 'num_open_accounts',
                'num_delinquencies', 'utilization']
TARGET = 'default'

print(f'数据维度：{df.shape}')
print(f'违约率：{df[TARGET].mean():.2%}')
print(f'违约人数：{df[TARGET].sum()} / {len(df)}')
print()
df.describe().round(3)

## 16.1 探索性分析（EDA）

信用数据的 EDA 重点关注两点：
1. **违约率分布**：整体违约率、按特征分组的违约率
2. **特征分布差异**：好客户（default=0）与坏客户（default=1）的特征分布是否有明显区别

特征分布分离程度越大，该特征越可能具有预测能力（高 IV）。

In [ ]:
# Cell 2：EDA —— 特征分布对比（好客户 vs 坏客户）
good = df[df[TARGET] == 0]
bad  = df[df[TARGET] == 1]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

feature_labels = {
    'age': '年龄（岁）',
    'income': '年收入（元）',
    'debt_to_income': '负债收入比',
    'credit_history_months': '信用历史（月）',
    'num_open_accounts': '未结清账户数',
    'num_delinquencies': '逾期记录次数',
    'utilization': '信用卡使用率'
}

for i, feat in enumerate(FEATURE_COLS):
    ax = axes[i]
    ax.hist(good[feat], bins=30, alpha=0.6, color='steelblue', label='正常(0)', density=True)
    ax.hist(bad[feat],  bins=30, alpha=0.6, color='tomato',    label='违约(1)', density=True)
    ax.set_title(feature_labels[feat], fontsize=11)
    ax.set_ylabel('密度')
    ax.legend(fontsize=8)

# 最后一格：整体违约率饼图
ax = axes[7]
n_bad  = df[TARGET].sum()
n_good = len(df) - n_bad
ax.pie([n_good, n_bad], labels=[f'正常\n({n_good})', f'违约\n({n_bad})'],
       colors=['steelblue', 'tomato'], autopct='%1.1f%%', startangle=90)
ax.set_title(f'样本分布（违约率={n_bad/len(df):.1%}）')

plt.suptitle('信用数据特征分布：好客户 vs 坏客户', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

# 按特征分组的违约率（简单展示）
print('各特征均值对比（好客户 vs 坏客户）：')
compare = pd.DataFrame({
    '正常客户均值': good[FEATURE_COLS].mean(),
    '违约客户均值': bad[FEATURE_COLS].mean(),
}).round(3)
compare['差异方向'] = compare.apply(
    lambda r: '↑违约风险更高' if r['违约客户均值'] > r['正常客户均值'] else '↓违约风险更低', axis=1)
print(compare.to_string())

## 16.2 WOE/IV 计算与变量筛选

**WOE（Weight of Evidence）** 公式：

$$WOE_i = \ln\frac{P(\text{Good}_i)}{P(\text{Bad}_i)} = \ln\frac{n_{\text{Good},i}/N_{\text{Good}}}{n_{\text{Bad},i}/N_{\text{Bad}}}$$

**IV（Information Value）** 公式：

$$IV = \sum_i (P(\text{Good}_i) - P(\text{Bad}_i)) \times WOE_i$$

IV 判断标准：< 0.02 无用，0.02-0.1 弱，0.1-0.3 中等，0.3-0.5 强，> 0.5 可疑。

In [ ]:
# Cell 3：WOE/IV 手工实现

def calc_woe_iv(df, feature, target, n_bins=5, method='quantile'):
    """
    计算连续特征的 WOE 和 IV。
    method: 'quantile'（等频分箱）或 'uniform'（等宽分箱）
    返回: (woe_df, iv_value)
    """
    df_work = df[[feature, target]].copy()

    # 分箱
    if method == 'quantile':
        df_work['bin'] = pd.qcut(df_work[feature], q=n_bins, duplicates='drop')
    else:
        df_work['bin'] = pd.cut(df_work[feature], bins=n_bins)

    N_good = (df_work[target] == 0).sum()
    N_bad  = (df_work[target] == 1).sum()

    grouped = df_work.groupby('bin', observed=True)[target].agg(
        n_bad=lambda x: (x == 1).sum(),
        n_good=lambda x: (x == 0).sum(),
        total='count'
    ).reset_index()

    # 避免零值（加极小值）
    eps = 0.5
    grouped['p_good'] = (grouped['n_good'] + eps) / (N_good + eps * len(grouped))
    grouped['p_bad']  = (grouped['n_bad']  + eps) / (N_bad  + eps * len(grouped))
    grouped['woe']    = np.log(grouped['p_good'] / grouped['p_bad'])
    grouped['iv_i']   = (grouped['p_good'] - grouped['p_bad']) * grouped['woe']
    grouped['bad_rate'] = grouped['n_bad'] / grouped['total']

    iv = grouped['iv_i'].sum()
    return grouped, iv


def iv_strength(iv):
    if iv < 0.02:   return '极弱（可丢弃）'
    elif iv < 0.10: return '弱'
    elif iv < 0.30: return '中等'
    elif iv < 0.50: return '强'
    else:           return '极强（疑似泄露）'


# 计算所有特征的 IV
iv_results = []
for feat in FEATURE_COLS:
    _, iv = calc_woe_iv(df, feat, TARGET, n_bins=5)
    iv_results.append({'特征': feat, 'IV': round(iv, 4), '预测能力': iv_strength(iv)})

iv_df = pd.DataFrame(iv_results).sort_values('IV', ascending=False)
print('=== 各特征 IV 汇总 ===')
print(iv_df.to_string(index=False))

# 可视化
fig, ax = plt.subplots(figsize=(9, 4))
colors = ['#d62728' if iv > 0.3 else '#1f77b4' if iv > 0.1 else '#aec7e8'
          for iv in iv_df['IV']]
ax.barh(iv_df['特征'][::-1], iv_df['IV'][::-1], color=colors[::-1], alpha=0.85)
ax.axvline(0.02, color='gray',   linestyle=':', lw=1.5, label='0.02（极弱线）')
ax.axvline(0.10, color='orange', linestyle='--', lw=1.5, label='0.10（弱→中 分界）')
ax.axvline(0.30, color='green',  linestyle='--', lw=1.5, label='0.30（中→强 分界）')
for bar, val in zip(ax.patches, iv_df['IV'][::-1]):
    ax.text(val + 0.003, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=9)
ax.set_xlabel('IV（信息值）')
ax.set_title('各特征 IV —— 变量预测能力排名')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 16.3 WOE 分箱可视化

对 IV 最高的两个特征做详细分箱分析，验证：
1. WOE 的单调性（与业务直觉一致）
2. 各箱的违约率趋势
3. 分箱是否合理（每箱有足够样本）

In [ ]:
# Cell 4：WOE 分箱可视化（选 IV 最高的两个特征）
top2_feats = iv_df['特征'].tolist()[:2]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for row, feat in enumerate(top2_feats):
    woe_table, iv_val = calc_woe_iv(df, feat, TARGET, n_bins=5)
    bins_label = [str(b) for b in woe_table['bin']]

    # 左图：WOE 柱状图
    ax_woe = axes[row][0]
    bar_colors = ['tomato' if w < 0 else 'steelblue' for w in woe_table['woe']]
    ax_woe.bar(range(len(bins_label)), woe_table['woe'], color=bar_colors, alpha=0.85)
    ax_woe.axhline(0, color='black', lw=0.8)
    ax_woe.set_xticks(range(len(bins_label)))
    ax_woe.set_xticklabels(bins_label, rotation=25, ha='right', fontsize=8)
    ax_woe.set_ylabel('WOE')
    ax_woe.set_title(f'{feat}  —  WOE（IV={iv_val:.4f}）')
    for xi, (woe_v, n) in enumerate(zip(woe_table['woe'], woe_table['total'])):
        ax_woe.text(xi, woe_v + 0.02 * np.sign(woe_v), f'n={n}', ha='center', fontsize=8)

    # 右图：各箱违约率
    ax_dr = axes[row][1]
    ax_dr.bar(range(len(bins_label)), woe_table['bad_rate'] * 100,
              color=['tomato' if r > df[TARGET].mean() else 'steelblue'
                     for r in woe_table['bad_rate']], alpha=0.85)
    ax_dr.axhline(df[TARGET].mean() * 100, color='black', linestyle='--',
                  lw=1.5, label=f'整体违约率 {df[TARGET].mean():.1%}')
    ax_dr.set_xticks(range(len(bins_label)))
    ax_dr.set_xticklabels(bins_label, rotation=25, ha='right', fontsize=8)
    ax_dr.set_ylabel('违约率（%）')
    ax_dr.set_title(f'{feat}  —  各箱违约率')
    ax_dr.yaxis.set_major_formatter(mtick.FormatStrFormatter('%.1f%%'))
    ax_dr.legend(fontsize=9)
    for xi, r in enumerate(woe_table['bad_rate']):
        ax_dr.text(xi, r * 100 + 0.2, f'{r:.1%}', ha='center', fontsize=8)

plt.suptitle('WOE 分箱分析：WOE 值与各箱违约率', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

# 打印详细分箱表
for feat in top2_feats:
    woe_table, iv_val = calc_woe_iv(df, feat, TARGET, n_bins=5)
    print(f'\n=== {feat}（IV={iv_val:.4f}）===')
    display_cols = ['bin', 'n_good', 'n_bad', 'total', 'bad_rate', 'woe', 'iv_i']
    print(woe_table[display_cols].round(4).to_string(index=False))

## 16.4 训练集/测试集划分 + WOE 编码

使用分层抽样（`stratify=y`）保持训练集和测试集的违约率一致，然后对所有特征做 WOE 编码，再送入逻辑回归。

> **注意**：WOE 分箱参数必须在**训练集**上估计，再应用到测试集，避免数据泄露。

In [ ]:
# Cell 5：训练/测试划分 + WOE 编码
X = df[FEATURE_COLS]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)
print(f'训练集：{len(X_train)} 条，违约率={y_train.mean():.2%}')
print(f'测试集：{len(X_test)}  条，违约率={y_test.mean():.2%}')

# 在训练集上计算 WOE，再转换测试集
train_df = X_train.copy()
train_df[TARGET] = y_train.values

woe_maps = {}  # 存储每个特征的 bin→WOE 映射

def woe_transform(df_src, feature, woe_map_series, n_bins=5):
    """用等频分箱的 bin 区间将原始特征映射到 WOE 值"""
    # 用 pd.qcut 返回 Categorical 的区间，再 map
    _, bin_edges = pd.qcut(df_src[feature], q=n_bins, duplicates='drop', retbins=True)
    bin_edges[0]  = -np.inf
    bin_edges[-1] = np.inf
    bins_cut = pd.cut(df_src[feature], bins=bin_edges)
    return bins_cut.map(woe_map_series).fillna(0).values


# 构建训练集 WOE 编码矩阵
X_train_woe = np.zeros((len(X_train), len(FEATURE_COLS)))
X_test_woe  = np.zeros((len(X_test),  len(FEATURE_COLS)))

for j, feat in enumerate(FEATURE_COLS):
    woe_tbl, _ = calc_woe_iv(train_df, feat, TARGET, n_bins=5)
    # bin → WOE 映射
    woe_map = pd.Series(woe_tbl['woe'].values, index=woe_tbl['bin'].values)

    # 重新计算 bin edges（在训练集上）
    try:
        _, bin_edges = pd.qcut(train_df[feat], q=5, duplicates='drop', retbins=True)
    except ValueError:
        _, bin_edges = pd.qcut(train_df[feat], q=3, duplicates='drop', retbins=True)
    bin_edges[0]  = -np.inf
    bin_edges[-1] = np.inf

    woe_maps[feat] = (woe_map, bin_edges)

    train_bins = pd.cut(X_train[feat], bins=bin_edges)
    test_bins  = pd.cut(X_test[feat],  bins=bin_edges)
    X_train_woe[:, j] = train_bins.map(woe_map).fillna(0).values
    X_test_woe[:,  j] = test_bins.map(woe_map).fillna(0).values

print('\nWOE 编码完成。训练集 WOE 矩阵预览（前3行）：')
pd.DataFrame(X_train_woe[:3], columns=FEATURE_COLS).round(4)

## 16.5 逻辑回归评分卡训练

在 WOE 编码后的特征上训练逻辑回归。由于 WOE 已经将特征统一到对数几率刻度，
无需额外标准化，但仍建议使用 L2 正则化（`C` 参数）防止过拟合。

逻辑回归系数 $\beta_j$ 直接表示第 $j$ 个特征的 WOE 值每增加 1 单位，
log-odds 的变化量。若 WOE 单调，则 $\beta_j > 0$ 意味着该特征正向驱动违约风险。

In [ ]:
# Cell 6：逻辑回归训练与系数解读
lr_model = LogisticRegression(C=1.0, max_iter=500, random_state=SEED)
lr_model.fit(X_train_woe, y_train)

# 系数表
coef_df = pd.DataFrame({
    '特征': FEATURE_COLS,
    '系数 β': lr_model.coef_[0].round(4)
}).sort_values('系数 β', ascending=False)
coef_df['方向'] = coef_df['系数 β'].apply(
    lambda b: '↑ 正向（WOE↑ → 违约概率↑）' if b > 0 else '↓ 负向（WOE↑ → 违约概率↓）'
)
print(f'截距 β₀ = {lr_model.intercept_[0]:.4f}')
print()
print('=== 逻辑回归系数 ===')
print(coef_df.to_string(index=False))

# 可视化系数
fig, ax = plt.subplots(figsize=(9, 5))
sorted_feats = coef_df['特征'].tolist()
sorted_coefs = coef_df['系数 β'].tolist()
bar_cols = ['#d62728' if c > 0 else '#1f77b4' for c in sorted_coefs]
ax.barh(sorted_feats[::-1], sorted_coefs[::-1], color=bar_cols[::-1], alpha=0.85)
ax.axvline(0, color='black', lw=0.8)
ax.set_xlabel('逻辑回归系数 β（WOE 特征）')
ax.set_title('逻辑回归评分卡：各特征系数')
for bar, val in zip(ax.patches, sorted_coefs[::-1]):
    offset = 0.01 if val >= 0 else -0.01
    ha = 'left' if val >= 0 else 'right'
    ax.text(val + offset, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', ha=ha, fontsize=9)
plt.tight_layout()
plt.show()

## 16.6 ROC/AUC + KS 曲线

**KS 曲线**的绘制步骤：
1. 将测试集按预测概率从低到高排序
2. 分别累积好客户和坏客户的比例（横轴为样本百分位）
3. 两条曲线差值最大处即 KS 统计量

KS 与 ROC 曲线的关系：KS 最大点对应 ROC 曲线上离对角线最远的点。

In [ ]:
# Cell 7：ROC/AUC + KS 曲线
y_prob = lr_model.predict_proba(X_test_woe)[:, 1]
auc_val = roc_auc_score(y_test, y_prob)
gini    = 2 * auc_val - 1

# KS 统计量计算
def calc_ks(y_true, y_prob):
    """计算 KS 统计量，返回 (ks值, ks对应阈值, 排序后的 good_cum, bad_cum, 百分位)"""
    df_ks = pd.DataFrame({'y': y_true, 'prob': y_prob})
    df_ks = df_ks.sort_values('prob').reset_index(drop=True)

    n_good = (df_ks['y'] == 0).sum()
    n_bad  = (df_ks['y'] == 1).sum()

    df_ks['cum_good'] = (df_ks['y'] == 0).cumsum() / n_good
    df_ks['cum_bad']  = (df_ks['y'] == 1).cumsum() / n_bad
    df_ks['diff']     = np.abs(df_ks['cum_bad'] - df_ks['cum_good'])

    ks_idx   = df_ks['diff'].idxmax()
    ks_val   = df_ks.loc[ks_idx, 'diff']
    ks_thresh = df_ks.loc[ks_idx, 'prob']
    pct = np.linspace(0, 1, len(df_ks))
    return ks_val, ks_thresh, df_ks['cum_good'].values, df_ks['cum_bad'].values, pct

ks_val, ks_thresh, cum_good, cum_bad, pct = calc_ks(y_test.values, y_prob)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左图：ROC 曲线
ax_roc = axes[0]
fpr, tpr, _ = roc_curve(y_test, y_prob)
ax_roc.plot(fpr, tpr, lw=2, color='#1f77b4', label=f'逻辑回归（AUC={auc_val:.4f}，Gini={gini:.4f}）')
ax_roc.plot([0, 1], [0, 1], 'k--', lw=1, label='随机猜测（AUC=0.5）')
ax_roc.fill_between(fpr, tpr, alpha=0.1, color='#1f77b4')
ax_roc.set_xlabel('假阳率（FPR）')
ax_roc.set_ylabel('真阳率（TPR）')
ax_roc.set_title('ROC 曲线')
ax_roc.legend(fontsize=9)

# 右图：KS 曲线
ax_ks = axes[1]
ax_ks.plot(pct, cum_bad,  lw=2, color='tomato',    label='坏客户累积分布（Bad CDF）')
ax_ks.plot(pct, cum_good, lw=2, color='steelblue', label='好客户累积分布（Good CDF）')
ks_pct_idx = np.argmax(np.abs(cum_bad - cum_good))
ax_ks.vlines(pct[ks_pct_idx], cum_good[ks_pct_idx], cum_bad[ks_pct_idx],
             color='green', linewidth=2.5,
             label=f'KS = {ks_val:.4f}（阈值≈{ks_thresh:.4f}）')
ax_ks.annotate(f'  KS={ks_val:.4f}',
               xy=(pct[ks_pct_idx], (cum_good[ks_pct_idx]+cum_bad[ks_pct_idx])/2),
               fontsize=10, color='green', fontweight='bold')
ax_ks.set_xlabel('样本百分位（按预测概率升序）')
ax_ks.set_ylabel('累积分布比例')
ax_ks.set_title('KS 曲线')
ax_ks.legend(fontsize=9)

plt.suptitle(f'模型评估：AUC={auc_val:.4f}，KS={ks_val:.4f}，Gini={gini:.4f}',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'\nAUC  = {auc_val:.4f}')
print(f'Gini = {gini:.4f}')
print(f'KS   = {ks_val:.4f}（阈值 ≈ {ks_thresh:.4f}）')

## 16.7 混淆矩阵与分类报告

以 KS 最大化对应的阈值作为决策阈值（而非默认的 0.5），
评估模型在测试集上的精确率、召回率和 F1 分数。

在信贷场景中，**漏判（FN）** 成本远高于**误判（FP）**，
因此坏客户召回率（Recall for class 1）是最关键的指标。

In [ ]:
# Cell 8：混淆矩阵与分类报告
# 使用 KS 阈值做分类
y_pred_ks  = (y_prob >= ks_thresh).astype(int)
y_pred_05  = (y_prob >= 0.5).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, y_pred, title in [
    (axes[0], y_pred_05, '默认阈值 0.5'),
    (axes[1], y_pred_ks, f'KS 最优阈值 ({ks_thresh:.3f})'),
]:
    ConfusionMatrixDisplay.from_predictions(
        y_test, y_pred,
        display_labels=['正常(0)', '违约(1)'],
        ax=ax, colorbar=False, cmap='Blues'
    )
    ax.set_title(title)

plt.suptitle('混淆矩阵对比：默认阈值 vs KS 最优阈值', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('=== 阈值=0.5 分类报告 ===')
print(classification_report(y_test, y_pred_05, target_names=['正常(0)', '违约(1)']))

print(f'=== 阈值={ks_thresh:.3f}（KS 最优）分类报告 ===')
print(classification_report(y_test, y_pred_ks, target_names=['正常(0)', '违约(1)']))

## 16.8 类别不平衡处理：class_weight 对比

`class_weight='balanced'` 自动将少数类（违约）的损失权重放大：

$$w_{\text{bad}} = \frac{N}{2 \times N_{\text{bad}}}, \quad w_{\text{good}} = \frac{N}{2 \times N_{\text{good}}}$$

这相当于在优化目标中，每个坏客户样本被计多次，迫使模型更努力地识别违约。

我们也演示**手工欠采样**（随机删减好客户至 1:1 比例）作为对比。

In [ ]:
# Cell 9：不平衡处理对比（原始 / class_weight / 欠采样）
from sklearn.metrics import recall_score, precision_score, f1_score

# --- 方案1：原始（无处理）---
lr_orig = LogisticRegression(C=1.0, max_iter=500, random_state=SEED)
lr_orig.fit(X_train_woe, y_train)
prob_orig = lr_orig.predict_proba(X_test_woe)[:, 1]

# --- 方案2：class_weight='balanced' ---
lr_cw = LogisticRegression(C=1.0, class_weight='balanced', max_iter=500, random_state=SEED)
lr_cw.fit(X_train_woe, y_train)
prob_cw = lr_cw.predict_proba(X_test_woe)[:, 1]

# --- 方案3：手工欠采样（随机删减好客户到 2:1 比例）---
rng_us = np.random.default_rng(SEED)
idx_bad  = np.where(y_train == 1)[0]
idx_good = np.where(y_train == 0)[0]
n_keep = min(len(idx_bad) * 2, len(idx_good))  # 保留至多 2 倍坏客户数
idx_good_kept = rng_us.choice(idx_good, size=n_keep, replace=False)
idx_under = np.concatenate([idx_bad, idx_good_kept])
rng_us.shuffle(idx_under)

X_under = X_train_woe[idx_under]
y_under = y_train.values[idx_under]

lr_us = LogisticRegression(C=1.0, max_iter=500, random_state=SEED)
lr_us.fit(X_under, y_under)
prob_us = lr_us.predict_proba(X_test_woe)[:, 1]

# 汇总（用各自的 0.5 阈值做分类）
results = []
for name, prob in [('原始（无处理）', prob_orig),
                    ('class_weight=balanced', prob_cw),
                    ('手工欠采样（2:1）', prob_us)]:
    pred = (prob >= 0.5).astype(int)
    results.append({
        '方案': name,
        'AUC': round(roc_auc_score(y_test, prob), 4),
        'KS': round(calc_ks(y_test.values, prob)[0], 4),
        '违约召回率': round(recall_score(y_test, pred, pos_label=1), 4),
        '违约精确率': round(precision_score(y_test, pred, pos_label=1, zero_division=0), 4),
        'F1（违约）': round(f1_score(y_test, pred, pos_label=1, zero_division=0), 4),
    })

res_df = pd.DataFrame(results)
print('=== 不平衡处理对比（阈值=0.5）===')
print(res_df.to_string(index=False))

# 可视化 ROC 对比
fig, ax = plt.subplots(figsize=(9, 5))
for (name, prob), color in zip(
    [('原始', prob_orig), ('class_weight=balanced', prob_cw), ('欠采样(2:1)', prob_us)],
    ['#1f77b4', '#d62728', '#2ca02c']
):
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc_v = roc_auc_score(y_test, prob)
    ax.plot(fpr, tpr, lw=2, color=color, label=f'{name}（AUC={auc_v:.4f}）')
ax.plot([0, 1], [0, 1], 'k--', lw=1)
ax.set_xlabel('假阳率（FPR）')
ax.set_ylabel('真阳率（TPR）')
ax.set_title('不平衡处理方案 ROC 曲线对比')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 16.9 评分卡刻度：概率转信用分

将模型输出的违约概率 $PD$ 转换为 300–850 风格的信用分：

$$\text{Score} = \text{Base} + PDO \times \log_2\left(\frac{\text{Odds}}{\text{BaseOdds}}\right)$$

其中 $\text{Odds} = \frac{1 - PD}{PD}$（好:坏比），高分对应低违约风险。

**参数设置（本示例）**：Base = 600，PDO = 20，BaseOdds = 50（即好:坏=50:1，约 2% 违约率）。

In [ ]:
# Cell 10：评分卡刻度转换
BASE_SCORE = 600
PDO        = 20
BASE_ODDS  = 50   # 好:坏 = 50:1

def prob_to_score(prob, base=BASE_SCORE, pdo=PDO, base_odds=BASE_ODDS):
    """将违约概率转换为信用评分（高分=低风险）"""
    odds = (1 - prob) / (prob + 1e-9)  # 好:坏 odds
    score = base + pdo * np.log2(odds / base_odds)
    return np.clip(score, 300, 850).round(0).astype(int)

# 转换测试集
scores = prob_to_score(prob_orig)

# 分数分布
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax1 = axes[0]
ax1.hist(scores[y_test == 0], bins=30, alpha=0.6, color='steelblue', label='正常客户')
ax1.hist(scores[y_test == 1], bins=30, alpha=0.6, color='tomato',    label='违约客户')
ax1.set_xlabel('信用评分')
ax1.set_ylabel('人数')
ax1.set_title('信用评分分布：正常客户 vs 违约客户')
ax1.legend()
ax1.axvline(580, color='black', linestyle='--', lw=1.5, label='截止分=580')

# 评分区间与违约率
ax2 = axes[1]
score_bins = [300, 500, 540, 560, 580, 600, 620, 650, 850]
score_labels = ['<500', '500-540', '540-560', '560-580', '580-600', '600-620', '620-650', '>650']
score_series = pd.Series(scores, name='score')
bin_cut = pd.cut(score_series, bins=score_bins, labels=score_labels)

dr_by_score = pd.DataFrame({'score_bin': bin_cut, 'default': y_test.values}).groupby(
    'score_bin', observed=True)['default'].agg(['mean', 'count']).reset_index()
dr_by_score.columns = ['区间', '违约率', '人数']

ax2.bar(range(len(dr_by_score)), dr_by_score['违约率'] * 100,
        color=['#d62728' if r > df[TARGET].mean() else '#1f77b4'
               for r in dr_by_score['违约率']], alpha=0.85)
ax2.axhline(df[TARGET].mean() * 100, color='black', linestyle='--', lw=1.5,
            label=f'整体违约率 {df[TARGET].mean():.1%}')
ax2.set_xticks(range(len(dr_by_score)))
ax2.set_xticklabels(dr_by_score['区间'], rotation=30, ha='right', fontsize=9)
ax2.set_ylabel('违约率（%）')
ax2.set_title('信用评分区间 vs 违约率')
ax2.yaxis.set_major_formatter(mtick.FormatStrFormatter('%.1f%%'))
ax2.legend()
for xi, (r, n) in enumerate(zip(dr_by_score['违约率'], dr_by_score['人数'])):
    ax2.text(xi, r*100+0.2, f'{r:.1%}\n(n={n})', ha='center', fontsize=8)

plt.suptitle('评分卡刻度：概率 → 信用分（Base=600，PDO=20）', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# 打印刻度对照表
print('\n评分刻度对照表（Base=600，PDO=20，BaseOdds=50）：')
demo_pd = [0.005, 0.01, 0.02, 0.05, 0.10, 0.20, 0.30, 0.50]
demo_table = pd.DataFrame({
    'PD': demo_pd,
    'Odds(好:坏)': [round((1-p)/p, 1) for p in demo_pd],
    '信用分': [prob_to_score(p) for p in demo_pd]
})
print(demo_table.to_string(index=False))

## 16.10 XGBoost 对比（可选）

XGBoost 不依赖 WOE 分箱，可直接使用原始特征，能自动捕获非线性与交互效应。

本节与逻辑回归（WOE 编码）在 AUC 和 KS 上做直接比较，
并展示 XGBoost 的特征重要性（Gain）作为可解释性辅助。

In [ ]:
# Cell 11：XGBoost 对比
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print('XGBoost 未安装，跳过此 Cell')

if HAS_XGB:
    # 直接使用原始特征（无需 WOE 编码）
    X_tr_raw = X_train.values
    X_te_raw = X_test.values

    xgb_model = XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=2.0,
        scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),  # 处理不平衡
        eval_metric='logloss',
        random_state=SEED,
        verbosity=0,
    )
    xgb_model.fit(X_tr_raw, y_train)
    prob_xgb = xgb_model.predict_proba(X_te_raw)[:, 1]

    auc_xgb = roc_auc_score(y_test, prob_xgb)
    ks_xgb  = calc_ks(y_test.values, prob_xgb)[0]
    auc_lr  = roc_auc_score(y_test, prob_orig)
    ks_lr   = calc_ks(y_test.values, prob_orig)[0]

    print('=== 模型对比 ===')
    print(f'逻辑回归（WOE）  AUC={auc_lr:.4f}  KS={ks_lr:.4f}')
    print(f'XGBoost（原始）  AUC={auc_xgb:.4f}  KS={ks_xgb:.4f}')

    # ROC 对比图 + 特征重要性
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # ROC
    ax_roc = axes[0]
    for prob_v, name, color in [
        (prob_orig, f'逻辑回归（AUC={auc_lr:.4f}）', '#1f77b4'),
        (prob_xgb,  f'XGBoost（AUC={auc_xgb:.4f}）', '#d62728'),
    ]:
        fpr, tpr, _ = roc_curve(y_test, prob_v)
        ax_roc.plot(fpr, tpr, lw=2, color=color, label=name)
    ax_roc.plot([0, 1], [0, 1], 'k--', lw=1)
    ax_roc.set_xlabel('假阳率')
    ax_roc.set_ylabel('真阳率')
    ax_roc.set_title('ROC 曲线：逻辑回归 vs XGBoost')
    ax_roc.legend(fontsize=10)

    # XGBoost 特征重要性（Gain）
    ax_imp = axes[1]
    imp = pd.Series(xgb_model.feature_importances_, index=FEATURE_COLS)
    imp = imp.sort_values()
    ax_imp.barh(imp.index, imp.values, color='#2ca02c', alpha=0.85)
    ax_imp.set_xlabel('特征重要性（Impurity）')
    ax_imp.set_title('XGBoost 特征重要性')
    for v, name in zip(imp.values, imp.index):
        ax_imp.text(v + 0.002, name, f'{v:.4f}', va='center', fontsize=9)

    plt.suptitle('XGBoost vs 逻辑回归评分卡', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

    # 综合汇总
    summary = pd.DataFrame([
        {'模型': '逻辑回归（WOE编码）', 'AUC': round(auc_lr, 4), 'KS': round(ks_lr, 4),
         'Gini': round(2*auc_lr-1, 4), '特点': '可解释、监管友好'},
        {'模型': 'XGBoost（原始特征）', 'AUC': round(auc_xgb, 4), 'KS': round(ks_xgb, 4),
         'Gini': round(2*auc_xgb-1, 4), '特点': '精度更高、需 SHAP 辅助解释'},
    ])
    print('\n=== 综合对比 ===')
    print(summary.to_string(index=False))

## 16.11 模型汇总

汇总全章建模结果，输出最终的评估指标对照表。

In [ ]:
# Cell 12：全章汇总
print('=' * 65)
print('第16章 信用风险评分卡建模 —— 结果汇总')
print('=' * 65)

all_results = []
for name, prob in [
    ('逻辑回归（WOE编码）', prob_orig),
    ('逻辑回归（WOE+class_weight）', prob_cw),
    ('逻辑回归（WOE+欠采样2:1）', prob_us),
]:
    ks_v = calc_ks(y_test.values, prob)[0]
    auc_v = roc_auc_score(y_test, prob)
    pred05 = (prob >= 0.5).astype(int)
    recall_v = recall_score(y_test, pred05, pos_label=1)
    all_results.append({
        '模型': name,
        'AUC': round(auc_v, 4),
        'KS': round(ks_v, 4),
        'Gini': round(2*auc_v-1, 4),
        '违约召回率@0.5': round(recall_v, 4),
    })

summary_df = pd.DataFrame(all_results)
print(summary_df.to_string(index=False))

print()
print('【主要发现】')
print('1. WOE 编码后的逻辑回归具有良好的区分能力（AUC/KS 均达到中等偏上水平）')
print('2. class_weight=balanced 主要提升坏客户召回率，对 AUC/KS 影响有限')
print('3. 欠采样会丢失正常客户信息，在当前数据集上效果与 class_weight 相近')
print('4. 最终部署建议：WOE 编码 + 逻辑回归，配合阈值调整满足业务召回率目标')

# 特征重要性排名（IV 角度）
print()
print('【特征重要性（IV 排名）】')
print(iv_df[['特征', 'IV', '预测能力']].to_string(index=False))

---
## 习题参考解答

以下 cell 对应正文习题，可直接运行验证。

In [ ]:
# === 习题 16.1：预期损失（EL）计算 ===
print('习题 16.1：预期损失计算')
print()
PD  = 0.03
LGD = 0.55
EAD = 50_000
EL1 = PD * LGD * EAD
print(f'PD={PD:.0%}, LGD={LGD:.0%}, EAD={EAD:,} 元')
print(f'EL = {PD:.0%} × {LGD:.0%} × {EAD:,} = {EL1:,.0f} 元')

PD2 = 0.02
EL2 = PD2 * LGD * EAD
print()
print(f'PD 降至 {PD2:.0%} 后：EL = {EL2:,.0f} 元，减少 {EL1-EL2:,.0f} 元（减少 {(EL1-EL2)/EL1:.1%}）')

# 可视化：PD 与 EL 的关系
pd_range = np.linspace(0.001, 0.30, 100)
el_range = pd_range * LGD * EAD
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(pd_range * 100, el_range, lw=2, color='tomato')
ax.axvline(PD * 100,  color='#1f77b4', linestyle='--', lw=1.5, label=f'PD={PD:.0%}，EL={EL1:,.0f}')
ax.axvline(PD2 * 100, color='green',   linestyle='--', lw=1.5, label=f'PD={PD2:.0%}，EL={EL2:,.0f}')
ax.set_xlabel('违约概率 PD（%）')
ax.set_ylabel('预期损失 EL（元）')
ax.set_title(f'预期损失 vs PD（LGD={LGD:.0%}，EAD={EAD:,}）')
ax.legend()
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.tight_layout()
plt.show()

In [ ]:
# === 习题 16.2：手工 WOE 计算 ===
print('习题 16.2：手工 WOE 计算')
print()

data_ex = pd.DataFrame({
    '分箱': ['低', '中', '高'],
    '好客户数': [4000, 3000, 1000],
    '坏客户数': [100, 250, 350],
})
N_good = data_ex['好客户数'].sum()
N_bad  = data_ex['坏客户数'].sum()

data_ex['p_good'] = data_ex['好客户数'] / N_good
data_ex['p_bad']  = data_ex['坏客户数'] / N_bad
data_ex['WOE']    = np.log(data_ex['p_good'] / data_ex['p_bad'])
data_ex['IV_i']   = (data_ex['p_good'] - data_ex['p_bad']) * data_ex['WOE']
data_ex['违约率'] = data_ex['坏客户数'] / (data_ex['好客户数'] + data_ex['坏客户数'])

IV_ex = data_ex['IV_i'].sum()
print(data_ex.round(4).to_string(index=False))
print(f'\nIV = {IV_ex:.4f} → {iv_strength(IV_ex)}')
print(f'WOE 趋势：{data_ex["WOE"].tolist()} → 单调递减 ✓')
print('解读：WOE 严格单调，与业务逻辑（越高风险越大）完全一致')

In [ ]:
# === 习题 16.3：KS 实践 + 两特征的 IV ===
print('习题 16.3：num_delinquencies 和 utilization 的 IV')
print()

for feat in ['num_delinquencies', 'utilization']:
    woe_tbl, iv_val = calc_woe_iv(df, feat, TARGET, n_bins=5)
    print(f'--- {feat} （IV={iv_val:.4f} {iv_strength(iv_val)}） ---')
    print(woe_tbl[['bin', 'n_bad', 'n_good', 'bad_rate', 'woe']].round(4).to_string(index=False))
    print()

print('\n习题 16.3（续）：debt_to_income 等频5箱违约率柱状图')
woe_dti, iv_dti = calc_woe_iv(df, 'debt_to_income', TARGET, n_bins=5)

fig, ax = plt.subplots(figsize=(9, 4))
bins_str = [str(b)[:15] for b in woe_dti['bin']]
bars = ax.bar(range(len(bins_str)), woe_dti['bad_rate']*100,
              color=['#d62728' if r > df[TARGET].mean() else '#1f77b4'
                     for r in woe_dti['bad_rate']])
ax.axhline(df[TARGET].mean() * 100, color='black', linestyle='--', lw=2,
           label=f'整体违约率 {df[TARGET].mean():.1%}')
ax.set_xticks(range(len(bins_str)))
ax.set_xticklabels(bins_str, rotation=20, ha='right', fontsize=9)
ax.set_ylabel('违约率（%）')
ax.set_title(f'debt_to_income 等频5箱违约率（IV={iv_dti:.4f}）')
ax.yaxis.set_major_formatter(mtick.FormatStrFormatter('%.1f%%'))
ax.legend()
for i, r in enumerate(woe_dti['bad_rate']):
    ax.text(i, r*100+0.2, f'{r:.1%}', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

mono_check = all(woe_dti['woe'].diff().dropna() <= 0)
print(f'WOE 单调性检验：{"单调递减 ✓" if mono_check else "不严格单调 ✗"}')
print(f'\n测试集 KS = {ks_val:.4f}（已在 Cell 7 中计算并在 KS 曲线上标注）')

In [ ]:
# === 习题 16.4：不平衡处理 —— 精确率/召回率权衡 ===
print('习题 16.4：class_weight 前后指标对比')
print()

detail_rows = []
for name, prob in [('原始（无处理）', prob_orig), ('class_weight=balanced', prob_cw)]:
    pred = (prob >= 0.5).astype(int)
    auc_v = roc_auc_score(y_test, prob)
    ks_v  = calc_ks(y_test.values, prob)[0]
    rec   = recall_score(y_test, pred, pos_label=1)
    prec  = precision_score(y_test, pred, pos_label=1, zero_division=0)
    f1_v  = f1_score(y_test, pred, pos_label=1, zero_division=0)
    detail_rows.append({
        '方案': name, 'AUC': round(auc_v,4), 'KS': round(ks_v,4),
        '召回率(违约)': round(rec,4), '精确率(违约)': round(prec,4), 'F1(违约)': round(f1_v,4)
    })

print(pd.DataFrame(detail_rows).to_string(index=False))
print()
print('分析：')
print('- AUC 和 KS 几乎不变（排序能力取决于概率分数，不受阈值影响）')
print('- class_weight 提高了坏客户召回率，但精确率下降（更多误报好客户）')
print('- 信贷实务：经济下行期，银行更倾向提高召回率（宁可误拒，不可漏判坏账）')
print('- 经济上行期，可适度降低召回率阈值，扩大通过率，追求规模增长')

In [ ]:
# === 习题 16.5：评分刻度计算 ===
print('习题 16.5：评分卡刻度公式验证')
print()
import math

BASE, PDO, BASEODDS = 600, 20, 50

# 验证 1：Odds=50 时应得 600 分
odds_base = 50
score_base = BASE + PDO * math.log2(odds_base / BASEODDS)
print(f'验证1：Odds=50 → Score = {BASE} + {PDO} × log₂({odds_base}/{BASEODDS}) = {score_base:.1f} ✓')
print()

# PD=8% 的得分
PD_ex = 0.08
odds_ex = (1 - PD_ex) / PD_ex
score_ex = BASE + PDO * math.log2(odds_ex / BASEODDS)
print(f'验证2：PD={PD_ex:.0%} → Odds = {odds_ex:.2f}')
print(f'       Score = {BASE} + {PDO} × log₂({odds_ex:.2f}/{BASEODDS}) = {score_ex:.1f}')
print(f'       截止分=580，该申请：{"通过 ✓" if score_ex >= 580 else "拒绝 ✗"}')
print()

# 可视化刻度曲线
pd_vals = np.linspace(0.005, 0.5, 200)
odds_vals = (1 - pd_vals) / pd_vals
score_vals = np.clip(BASE + PDO * np.log2(odds_vals / BASEODDS), 300, 850)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(pd_vals * 100, score_vals, lw=2, color='steelblue')
ax.axhline(580, color='tomato', linestyle='--', lw=1.5, label='截止分=580')
ax.axvline(PD_ex * 100, color='orange', linestyle='--', lw=1.5,
           label=f'PD={PD_ex:.0%}，Score={score_ex:.0f}')
ax.set_xlabel('违约概率 PD（%）')
ax.set_ylabel('信用评分')
ax.set_title('评分刻度曲线（Base=600，PDO=20，BaseOdds=50）')
ax.legend()
ax.invert_xaxis()  # 高分=低违约率，从右往左 PD 递增
plt.tight_layout()
plt.show()

# PDO 翻倍验证
odds_2x = BASEODDS * 2
score_2x = BASE + PDO * math.log2(odds_2x / BASEODDS)
print(f'PDO 验证：Odds翻倍（{BASEODDS}→{odds_2x}）时，Score = {score_2x:.0f}（增加了 {score_2x - BASE:.0f} = PDO={PDO} ✓）')